In [1]:
import $ivy.`dev.zio::izumi-reflect:3.0.6`, izumi.reflect.Tag

import $ivy.$                             , izumi.reflect.Tag

In [1]:
  /* Convert µDhall type expressions to Scala izumi-reflect's type tags (Tag typeclass) when possible. */
  // The type expression must be in the beta-normalized form: no "let"-expressions or type annotations may be left.
  // µDhall's `Type` is represented by Any.
  // Lambda expressions are not types and will not be supported for now.
  def asTag(e: Expr): Either[String, Tag[_]] = {
    e match {
      case Expr.Forall(name, tipe, body) =>
        for { // For now we do not support type variables or dependent types.
          tipeTag <- asTag(tipe)
          bodyTag <- asTag(body)
        } yield Tag.appliedTag(TagKK[Function1], List(tipeTag.tag, bodyTag.tag))

      case Expr.Builtin(Constant.Natural) => Right(Tag[Natural])
      case Expr.Builtin(Constant.Type)    => Right(Tag[Nothing])
      // case Expr.Applied(func, arg) => ???
      // case Expr.Lambda(name, tipe, body) => ???
      case _                              => Left(s"Cannot convert µDhall expression ${e.print} to a Scala type")
    }
  }

  Seq[(String, Tag[_])](
    "Natural"                         -> Tag[Natural],
    "Type"                            -> Tag[Nothing],
    "Natural -> Natural"              -> Tag[Natural => Natural],
    "Natural -> Natural -> Natural"   -> Tag[Natural => Natural => Natural],
    "(Natural -> Natural) -> Natural" -> Tag[(Natural => Natural) => Natural],
  ).validate(s => asTag(s.dhall).right.get)

  println("Tests passed for asTag().")

cmd2.sc:22: not found: type Natural
    "Natural"                         -> Tag[Natural],
                                             ^
cmd2.sc:24: not found: type Natural
    "Natural -> Natural"              -> Tag[Natural => Natural],
                                             ^
cmd2.sc:24: not found: type Natural
    "Natural -> Natural"              -> Tag[Natural => Natural],
                                                        ^
cmd2.sc:25: not found: type Natural
    "Natural -> Natural -> Natural"   -> Tag[Natural => Natural => Natural],
                                             ^
cmd2.sc:25: not found: type Natural
    "Natural -> Natural -> Natural"   -> Tag[Natural => Natural => Natural],
                                                        ^
cmd2.sc:25: not found: type Natural
    "Natural -> Natural -> Natural"   -> Tag[Natural => Natural => Natural],
                                                                   ^
cmd2.sc:26: not found: type Natural
    

In [ ]:
  // This value models an expression translated from µDhall into Scala.
  // The expression still depends on some variables in an environment.
  // (Values of those variables are again expressions translated into Scala.)
  // The function scalaV will throw an exception if conversion is not possible.
  final case class ClosureV(scalaV: Map[Expr.Variable, ClosureV] => Any, source: Expr) {
    override def toString: String = s"${super.toString}[${Option(source).map(_.print).getOrElse("???")}]"

    def closedValue: Any = scalaV(Map())
  }

  def pureClosureV(x: Any): ClosureV = ClosureV(_ => x, null)

  // Cosmetic convenience: functions converted from µDhall can still print their Dhall source code.
  final case class FunctionV(f: Any => Any, source: Expr) extends (Any => Any) {
    override def toString: String = s"{ ${Option(source).map(_.print).getOrElse("???")} }"

    override def apply(v1: Any): Any = f(v1)
  }



  def asScala[A](e: Expr)(implicit tag: Tag[A]): A = (for {
    inferredType <- typeCheck(Nil, e)
    inferredTag  <- asTag(inferredType)
    _            <- if (inferredTag == tag) Right(())
                    else
                      Left(
                        s"Cannot convert from Dhall expression ${e.print} having incompatible type ${inferredType.print}, tag ${inferredTag}, to expected Scala type $tag"
                      )

  } yield convertValueToScala(e).closedValue.asInstanceOf[A]) match {
    case Left(error) =>
      throw new Exception(s"Cannot convert from Dhall expression ${e.print} to Scala type $tag: $error")
    case Right(x)    => x
  }

  private def convertValueToScala(e: Expr): ClosureV = {
    // newFunc must be computed as a closure.
    // Keep recursive calls to convertValueToScala() outside of the returned closure.
    val newFunc: Map[Expr.Variable, ClosureV] => Any = e match {
      case Expr.NaturalLiteral(value) => _ => value

      case v @ Expr.Variable(_, _)    =>
        scalaVars =>
          scalaVars.get(v) match {
            case Some(knownVariableAssignment) => knownVariableAssignment.scalaV(scalaVars)
            case None                          => throw new Exception(s"Undefined variable $v while known variables are $scalaVars")
          }

        // Example:
        //    "λ(n : Natural) → n + (λ(n : Natural) → n + n@1) 2" should evaluate to "λ(n : Natural) → n + 2 + n"
        // should be replaced by the Scala function:
      //    { x: Any => x.asInstanceOf[BigInt] + {x2 : Any => x2 + x}(2) }
      case Expr.Lambda(name, _, body) =>
        val bodyScala = convertValueToScala(body)

        { (scalaVars: Map[Expr.Variable, ClosureV]) =>
          val variables1 = shiftVars(up = true, name, scalaVars)

          FunctionV(
            { (varX: Any) =>
              val variables2 = variables1 ++ Map(Expr.Variable(name, 0) -> pureClosureV(varX))
              bodyScala.scalaV(variables2)
            },
            e,
          )
        }

      // Forall is always a function type, so we just convert it to a tag.
      // Only support non-dependent type signatures without type parameters.
      case Expr.Forall(_, _, _)       =>
        asTag(e) match {
          case Left(error) => throw new Exception("Cannot convert to Scala value: " + error)
          case Right(x)    => { _ => x }
        }

      case Expr.Let(name, subst, body) => // Evaluated as (λ(name) => body) subst.
        convertValueToScala(letExprAsApplied(name, subst, body)).scalaV

      case Expr.Annotated(body, _) => convertValueToScala(body).scalaV

      case Expr.Applied(func, arg) =>
        val functionHead = convertValueToScala(func)
        val argument     = convertValueToScala(arg)

        scalaVars => functionHead.scalaV(scalaVars).asInstanceOf[Function1[Any, Any]].apply(argument.scalaV(scalaVars))

      case Expr.Builtin(constant) =>
        val symbol: Any = constant match {
          case Constant.NaturalFold     => Natural_fold_native
          case Constant.NaturalSubtract => Natural_subtract_native
          case Constant.Natural         => Tag[Natural]
          case _                        => throw new Exception(s"Type symbol $constant cannot be converted to a Scala value")
        }
        _ => symbol

      case Expr.BinaryOp(left, op, right) =>
        // Helper function: apply a binary operation.
        // No type checking needed here, because all expressions were already type-checked.
        def useOp[P: Tag, Q: Tag, R: Tag](operator: (P, Q) => R)(scalaVars: Map[Expr.Variable, ClosureV]) = {
          val evalLop = convertValueToScala(left)
          val evalRop = convertValueToScala(right)
          operator(evalLop.scalaV(scalaVars).asInstanceOf[P], evalRop.scalaV(scalaVars).asInstanceOf[Q])
        }

        op match {
          case Operator.Plus  => useOp[Natural, Natural, Natural](_ + _)
          case Operator.Times => useOp[Natural, Natural, Natural](_ * _)
        }
    }
    ClosureV(newFunc, e)
  }

  // Helper function: apply shift() to all variables in a given dictionary.
  def shiftVars[T](up: Boolean, varName: String, vars: Map[Expr.Variable, T]): Map[Expr.Variable, T] = vars.map { case (variable, value) =>
    shift(if (up) 1 else -1, varName, 0, variable).asInstanceOf[Expr.Variable] -> value // shift() transforms Variable into Variable.
  }

  // Helper function: replace "let name = subst in body" by "(λ(name) => body) subst". This is equivalent for evaluation (but not for type-checking!).
  def letExprAsApplied(name: String, subst: Expr, body: Expr): Expr =
    Expr.Applied(Expr.Lambda(name, subst.inferType, body), subst)

  // Public API.
  implicit class ExprAsScala(e: Expr) {
    def asScala[A: Tag]: A = MuDhall.asScala[A](e)
  }

  type Natural = Int

  val Natural_subtract_native: Natural => Natural => Natural = x => y => math.max(0, y - x)

  val Natural_fold_native: Natural => Tag[Nothing] => (Any => Any) => Any => Any = { m => _ => update => init =>
    @tailrec
    def loop(currentResult: Any, counter: Natural): Any =
      if (counter >= m) currentResult
      else {
        val newResult = update(currentResult)
        if (newResult == currentResult) {
          // Shortcut: the result did not change after applying `update`, so no need to continue the loop.
          currentResult
        } else {
          loop(newResult, counter + 1)
        }
      }
    loop(currentResult = init, counter = 0)
  }

  Seq[(String, Natural)](
    "12345"                                                    -> 12345,
    "1 + 2 + 3 + 4"                                            -> 10,
    "1 * 2 * 3 * 4"                                            -> 24,
    "Natural/subtract 1 1000"                                  -> 999,
    "(λ(n : Natural) → n + 1) 1000"                            -> 1001,
    "(λ(n : Natural) → n + 1) ((λ(n : Natural) → n + 1) 1000)" -> 1002,
    "Natural/fold 10 Natural (λ(n : Natural) → n + 1) 100"     -> 110,
  ).validate(s => s.dhall.asScala[Natural])

  expect(
    Try(
      "λ(n : Natural) → n".dhall.asScala[Natural => Boolean]
    ).failed.get.getMessage == "Cannot convert from Dhall expression λ(n : Natural) → n to Scala type Tag[Function1[-Int,+Boolean]]: Cannot convert from Dhall expression λ(n : Natural) → n having incompatible type ∀(n : Natural) → Natural, tag Tag[Function1[-Int,+Int]], to expected Scala type Tag[Function1[-Int,+Boolean]]"
  )

  // Convert a function of type Natural → Natural from µDhall to a Scala function of type Int => Int.
  // Then apply that Scala function to a Scala argument of type Int. Verify the expected result.
  Seq[(String, Natural, Natural)](
    ("λ(n : Natural) → n", 10, 10),
    ("λ(n : Natural) → n + 20", 10, 30),
    ("λ(n : Natural) → 3 * n + (λ(n : Natural) → n + n@1) 2", 10, 42),
    ("(λ(n : Natural) → λ(a : Natural) → a + n) 100", 20, 120),
    ("Natural/subtract 2", 10, 8),
    ("Natural/subtract 20", 10, 0),
    ("(λ(f : Natural → Natural) → λ(x : Natural) → x + f x) (λ(a : Natural) → a + 1)", 10, 21),
  ).zipWithIndex.foreach { case ((funcMuDhall, x, expected), i) =>
    val f: Natural => Natural = funcMuDhall.dhall.asScala[Natural => Natural]
    expect(i >= 0 && f(x) == expected)
  }

  val x: Tag[Natural => Natural] = Tag[Natural => Natural]
  expect("Natural → Natural".dhall.asScala == Tag[Natural => Natural])

  val g1      = "λ(f : Natural → Natural) → λ(n : Natural) → 1 + f n".dhall
  val g1scala = g1.asScala[(Natural => Natural) => Natural => Natural]
  expect(g1scala.toString == "{ λ(f : ∀(_ : Natural) → Natural) → λ(n : Natural) → 1 + f n }")

  val g2      = "λ(n : Natural) → n + 2".dhall
  val g2scala = g2.asScala[Natural => Natural]
  expect(g2scala.toString == "{ λ(n : Natural) → n + 2 }")
  expect(g2scala(2) == 4)
  val g3      = g1(g2)
  val g3scala = g3.asScala[Natural => Natural]

  expect(g3scala.toString == "{ λ(n : Natural) → 1 + f n }")
  expect(g3scala(2) == 5)
  val g3scalaByScala = g1scala(g2scala)
  expect(g3scalaByScala(2) == 5)
  val g4scalaByScala = g1scala(g3scalaByScala)
  expect(g4scalaByScala(2) == 6)
  val foldByScala    = Natural_fold_native(3)(Tag[Nothing])(g1scala.asInstanceOf[Any => Any])(g2scala).asInstanceOf[Any => Any]
  expect(foldByScala(10) == 15)

  println("A loop involving a function type.")
  val fDhall = "Natural/fold 3 (Natural → Natural) (λ(f : Natural → Natural) → λ(n : Natural) → 1 + f n) (λ(n : Natural) → n + 2)".dhall
  val fScala = fDhall.asScala[Natural => Natural]
  expect(fScala(10) == 15)

  println("Tests passed for asScala().")